# 04 — Process DB-COMP documents

## Purpose

This notebook converts the raw DB-COMP PDF collection created by `03_download_dbcomp2.ipynb`
into a canonical plain-text representation for later screening, metadata extraction and
dataset construction.

The processing logic is intentionally simple enough to describe directly in a Scientific
Data Methods section:

1. load the DB-COMP manifest produced by notebook 03;
2. verify that the referenced raw PDFs exist;
3. extract the embedded PDF text with **PyMuPDF**;
4. if that extraction looks technically suspicious, try **Poppler `pdftotext -layout`**;
5. if the second native extraction is also suspicious, run **MinerU OCR** explicitly with the `pipeline` backend and OCR mode;\n6. retain the first extraction that passes the technical checks; if none passes but a
   non-empty extraction exists, retain the last available extraction;
7. apply only light text normalisation and write one UTF-8 `.txt` file.

The notebook does not classify the legal content of documents, infer Commission decision
dates, reconcile DB-COMP records with other Commission sources, or create text specifically
flattened for regex or LLM use. Those steps belong elsewhere in the pipeline.

Detailed candidate scoring used in earlier development versions has been removed.
Fallback decisions are based only on a small set of transparent technical checks.

## Inputs and outputs

The input is the compact manifest produced by notebook 03:

`output/com_dbcomp/db_comp_manifest.csv`

Raw PDFs remain unchanged under:

`data/raw/com_dbcomp/files/`

For every successfully processed PDF, this notebook writes one canonical text file under:

`data/processed/com_dbcomp/text/`

When the source PDF filename is unique, the processed text keeps the original filename stem. For example:

- `ec_cjeu/data/raw/com_dbcomp/files/example.pdf`
- `ec_cjeu/data/processed/com_dbcomp/text/example.txt`

DB-COMP can contain different documents whose remote PDFs have the same filename. In that
case, the processed text filename receives a short deterministic hash derived from the unique
`file_url`, for example `example__a1b2c3d4e5f6.txt`. This prevents one document from
overwriting another while keeping the naming stable across reruns.

Paths written to the processing manifest and JSONL state are portable paths beginning with
`ec_cjeu/`; no username, drive letter or machine-specific parent directory is stored.

Processing state is written incrementally to:

`logs/com_dbcomp/db_comp_processing_state.jsonl`

The publication-facing processing manifest is:

`output/com_dbcomp/db_comp_text_manifest.csv`


In [1]:
from __future__ import annotations

import hashlib
import json
import os
import re
import shutil
import subprocess
import tempfile
import unicodedata
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import fitz  # PyMuPDF
import pandas as pd
from tqdm.auto import tqdm


# ---------------------------------------------------------------------
# Project paths
# ---------------------------------------------------------------------

REPOSITORY_LABEL = "ec_cjeu"


def find_project_root(start: Path | None = None) -> Path:
    """Locate the repository root from the code/01_commission layout."""
    override = os.environ.get("ECCJEU_PROJECT_ROOT", "").strip()
    if override:
        root = Path(override).expanduser().resolve()
        if not root.exists():
            raise FileNotFoundError(f"ECCJEU_PROJECT_ROOT does not exist: {root}")
        return root

    start = (start or Path.cwd()).resolve()

    for candidate in [start, *start.parents]:
        if (candidate / "code" / "01_commission").exists():
            return candidate

    raise FileNotFoundError(
        "Could not locate the project root. Run the notebook from within the "
        "ec_cjeu repository or set ECCJEU_PROJECT_ROOT."
    )


PROJECT_ROOT = find_project_root()

INPUT_MANIFEST = PROJECT_ROOT / "output" / "com_dbcomp" / "db_comp_manifest.csv"
RAW_PDF_DIR = PROJECT_ROOT / "data" / "raw" / "com_dbcomp" / "files"
PROCESSED_TEXT_DIR = PROJECT_ROOT / "data" / "processed" / "com_dbcomp" / "text"
LOG_DIR = PROJECT_ROOT / "logs" / "com_dbcomp"
OUTPUT_DIR = PROJECT_ROOT / "output" / "com_dbcomp"

PROCESSING_STATE_JSONL = LOG_DIR / "db_comp_processing_state.jsonl"
TEXT_MANIFEST_CSV = OUTPUT_DIR / "db_comp_text_manifest.csv"

for directory in [PROCESSED_TEXT_DIR, LOG_DIR, OUTPUT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------------------
# Processing configuration
# ---------------------------------------------------------------------

PROCESSING_VERSION = "1.0"
FORCE_REPROCESS = False

RUN_PDFTOTEXT = True
RUN_OCR = True

# MinerU OCR runtime controls.
# The outer subprocess timeout scales with PDF length, while MinerU's own
# internal timeouts are also raised so they do not terminate long scans first.
OCR_TIMEOUT_SECONDS = 1800          # minimum: 30 minutes
OCR_SECONDS_PER_PAGE = 60
OCR_TIMEOUT_CAP_SECONDS = 7200      # maximum: 2 hours

MINERU_PDF_RENDER_TIMEOUT_SECONDS = 1800
MINERU_LOCAL_API_STARTUP_TIMEOUT_SECONDS = 600
MINERU_TASK_RESULT_TIMEOUT_SECONDS = 7200
MINERU_PROCESSING_WINDOW_SIZE = 32

# OCR is only called after both native extractors have failed their checks,
# so use the lightweight/general pipeline backend and force OCR explicitly.
MINERU_FORCE_OCR_MODE = True
MINERU_PREFER_PIPELINE_BACKEND = True

# Simple technical checks used only to decide whether another extractor is needed.
MIN_TOTAL_CHARS = 200
MIN_PDF_CHARS_PER_PAGE = 500
MAX_BAD_ENCODING_MARKERS = 3
MAX_ONE_CHAR_TOKEN_RATIO = 0.20

BAD_ENCODING_MARKERS = [
    "�",
    "Ã",
    "Â",
    "â€™",
    "â€œ",
    "â€",
    "ðŸ",
]

PDFTOTEXT_CLI = shutil.which("pdftotext")
MINERU_CLI = shutil.which("mineru") or shutil.which("magic-pdf")


def repo_path(path: Path) -> str:
    """Portable path shown as ec_cjeu/<path inside repository>."""
    relative = path.resolve().relative_to(PROJECT_ROOT.resolve()).as_posix()
    return f"{REPOSITORY_LABEL}/{relative}"


def resolve_manifest_path(value: str) -> Path:
    """
    Resolve either data/... or ec_cjeu/data/... paths from notebook 03.
    """
    value = str(value or "").strip().replace("\\", "/")
    if not value:
        raise ValueError("Empty repository path.")

    parts = Path(value).parts
    if parts and parts[0] == REPOSITORY_LABEL:
        value = Path(*parts[1:]).as_posix()

    path = Path(value)
    if path.is_absolute() or ".." in path.parts or re.match(r"^[A-Za-z]:[/\\]", value):
        raise ValueError(f"Non-portable path: {value}")

    return PROJECT_ROOT / path


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat()


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


print("Project root: .")
print("Input manifest: output/com_dbcomp/db_comp_manifest.csv")
print("Raw PDFs: data/raw/com_dbcomp/files/")
print("Processed text: data/processed/com_dbcomp/text/")
print("Processing state: logs/com_dbcomp/db_comp_processing_state.jsonl")
print("Output manifest: output/com_dbcomp/db_comp_text_manifest.csv")
print("pdftotext available:", PDFTOTEXT_CLI is not None)
print("MinerU available:", MINERU_CLI is not None)

Project root: .
Input manifest: output/com_dbcomp/db_comp_manifest.csv
Raw PDFs: data/raw/com_dbcomp/files/
Processed text: data/processed/com_dbcomp/text/
Processing state: logs/com_dbcomp/db_comp_processing_state.jsonl
Output manifest: output/com_dbcomp/db_comp_text_manifest.csv
pdftotext available: True
MinerU available: True


## 1. Validate the handoff from notebook 03

The DB-COMP manifest is treated as the source inventory for this processing step.

Before extraction, the notebook checks only what is needed for a reliable handoff:

- required manifest columns are present;
- DB-COMP download URLs are unique and non-empty;
- source filenames are non-empty (duplicate remote filenames are allowed);
- every row is marked as downloaded;
- every referenced raw file exists;
- all input documents are PDFs;
- processed text filenames are made unique deterministically when source filenames collide.

The raw PDF SHA-256 is calculated during processing and stored with the resulting text
record. This identifies the exact source file used without adding a separate audit table.


In [2]:
REQUIRED_COLUMNS = {
    "source",
    "dbcomp_document_id",
    "case_number",
    "case_number_source",
    "case_type",
    "dbcomp_date",
    "document_name",
    "file_url",
    "file_name",
    "downloaded",
    "local_path",
}


def read_manifest(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Required input is missing: {path}")
    return pd.read_csv(path, dtype=str, keep_default_na=False)


def build_unique_text_file_names(manifest: pd.DataFrame) -> list[str]:
    """
    Build deterministic output names without assuming remote PDF filenames are unique.

    Unique source filenames keep their original stem (example.pdf -> example.txt).
    If two or more rows would map to the same text filename, append a short SHA-256
    digest of the unique file_url so each source PDF gets a stable output path.
    """
    base_names = [
        Path(file_name).with_suffix(".txt").name
        for file_name in manifest["file_name"]
    ]

    # casefold() also protects against collisions on case-insensitive filesystems.
    normalized = pd.Series(base_names, dtype="string").str.casefold()
    colliding = normalized.duplicated(keep=False).tolist()

    output_names: list[str] = []
    for base_name, file_url, has_collision in zip(
        base_names,
        manifest["file_url"],
        colliding,
    ):
        if has_collision:
            digest = hashlib.sha256(file_url.encode("utf-8")).hexdigest()[:12]
            base_path = Path(base_name)
            output_name = f"{base_path.stem}__{digest}{base_path.suffix}"
        else:
            output_name = base_name
        output_names.append(output_name)

    normalized_output = [name.casefold() for name in output_names]
    if len(normalized_output) != len(set(normalized_output)):
        raise ValueError(
            "Could not construct unique processed text filenames from file_url."
        )

    return output_names


manifest_input = read_manifest(INPUT_MANIFEST)

missing_columns = sorted(REQUIRED_COLUMNS - set(manifest_input.columns))
if missing_columns:
    raise ValueError(f"DB-COMP manifest is missing required columns: {missing_columns}")

if manifest_input["file_url"].eq("").any() or not manifest_input["file_url"].is_unique:
    raise ValueError("file_url must be unique and non-empty.")

if manifest_input["file_name"].eq("").any():
    raise ValueError("file_name must be non-empty.")

downloaded = manifest_input["downloaded"].str.lower().isin({"true", "1", "yes"})
if not downloaded.all():
    missing = int((~downloaded).sum())
    raise ValueError(
        f"{missing} DB-COMP manifest row(s) are not marked as downloaded. "
        "Complete notebook 03 before processing."
    )

if not manifest_input["file_name"].str.lower().str.endswith(".pdf").all():
    raise ValueError("This DB-COMP processing notebook expects PDF inputs only.")

raw_paths = [
    resolve_manifest_path(value)
    for value in manifest_input["local_path"]
]

missing_files = [
    manifest_input.iloc[index]["file_name"]
    for index, path in enumerate(raw_paths)
    if not path.exists() or not path.is_file()
]
if missing_files:
    raise FileNotFoundError(
        f"{len(missing_files)} referenced raw PDF(s) are missing; "
        f"examples: {missing_files[:10]}"
    )

text_file_names = build_unique_text_file_names(manifest_input)

work_manifest = manifest_input.copy()
work_manifest["raw_path_internal"] = [str(path) for path in raw_paths]
work_manifest["text_file_name"] = text_file_names
work_manifest = work_manifest.sort_values(
    ["dbcomp_document_id", "file_name", "file_url"],
    na_position="last",
).reset_index(drop=True)

source_name_collisions = int(
    manifest_input["file_name"].str.casefold().duplicated(keep=False).sum()
)

print(f"Documents ready for processing: {len(work_manifest):,}")
if source_name_collisions:
    print(
        f"Rows with colliding source filenames: {source_name_collisions:,} "
        "(assigned deterministic hashed text filenames)"
    )

display(
    work_manifest[
        [
            "dbcomp_document_id",
            "case_number",
            "file_name",
            "text_file_name",
            "local_path",
        ]
    ].head()
)


Documents ready for processing: 837
Rows with colliding source filenames: 11 (assigned deterministic hashed text filenames)


,dbcomp_document_id,case_number,file_name,text_file_name,local_path
0,,39904,AT_39904_5942450_1110_7.pdf,AT_39904_5942450_1110_7.txt,data/raw/com_dbcomp/files/AT_39904_5942450_111...
1,,40135,AT_40135_8399795_5456_9.pdf,AT_40135_8399795_5456_9.txt,data/raw/com_dbcomp/files/AT_40135_8399795_545...
2,,40136,AT_40136_7970992_6095_3.pdf,AT_40136_7970992_6095_3.txt,data/raw/com_dbcomp/files/AT_40136_7970992_609...
3,,40490,AT_40490_8263501_702_3.pdf,AT_40490_8263501_702_3.txt,data/raw/com_dbcomp/files/AT_40490_8263501_702...
4,,40522,AT_40522_8986595_2868_7.pdf,AT_40522_8986595_2868_7__77389e4cd3db.txt,data/raw/com_dbcomp/files/AT_40522_8986595_286...


## 2. Text extraction strategy

PDF processing follows the same deterministic fallback hierarchy used for the legacy
Commission collection where that approach is applicable:

1. **PyMuPDF** sorted text is the default.
2. If the result is technically suspicious, **Poppler `pdftotext -layout`** is tried.
3. If the second native result is also suspicious, **MinerU OCR** is run.
4. The first extraction that passes the technical checks is retained.
5. If none passes but a method still produced non-empty text, the last available
   extraction is retained rather than silently dropping the document.
6. A document is marked `failed` only when no usable text can be produced at all.

There is no weighted quality score and no competition among several extraction candidates.
The checks below are used only to decide whether to continue down the fallback ladder.

### When is an extraction considered suspicious?

The checks are deliberately simple and technical:

- fewer than `MIN_TOTAL_CHARS` characters were extracted;
- the PDF contains fewer than `MIN_PDF_CHARS_PER_PAGE` extracted characters per page;
- at least `MAX_BAD_ENCODING_MARKERS` obvious encoding-corruption markers are present;
- more than `MAX_ONE_CHAR_TOKEN_RATIO` of alphabetic tokens contain only one character,
  which can indicate severely fragmented PDF text.

These thresholds are pragmatic extraction checks rather than measures of legal or linguistic
quality. They determine whether another extraction method should be attempted.

In [3]:
def normalize_unicode(text: str) -> str:
    text = unicodedata.normalize("NFKC", text or "")
    replacements = {
        "\u00a0": " ",
        "\u200b": "",
        "\ufeff": "",
        "\r\n": "\n",
        "\r": "\n",
        "\f": "\n\n",
    }
    for old, new in replacements.items():
        text = text.replace(old, new)
    return text


def clean_text(text: str) -> str:
    """
    Light normalisation only. Keep line and paragraph structure.
    """
    text = normalize_unicode(text)
    text = text.replace("\u00ad", "")

    # Join words broken by line-ending hyphenation.
    text = re.sub(
        r"(?<=[A-Za-zÀ-ÖØ-öø-ÿ])[-‐‑]\s*\n\s*(?=[a-zà-öø-ÿ])",
        "",
        text,
    )

    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r" *\n *", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip() + "\n" if text.strip() else ""


def count_bad_encoding_markers(text: str) -> int:
    return sum(text.count(marker) for marker in BAD_ENCODING_MARKERS)


def one_char_token_ratio(text: str) -> float:
    tokens = re.findall(r"[A-Za-zÀ-ÖØ-öø-ÿ]+", text or "")
    if not tokens:
        return 1.0
    return sum(len(token) == 1 for token in tokens) / len(tokens)


def assess_text(text: str, n_pages: int | None) -> dict[str, Any]:
    """Transparent technical checks used only to decide whether to fall back."""
    text = text or ""
    n_chars = len(text)
    chars_per_page = n_chars / n_pages if n_pages else None
    bad_marker_count = count_bad_encoding_markers(text)
    fragment_ratio = one_char_token_ratio(text)

    reasons: list[str] = []

    if n_chars < MIN_TOTAL_CHARS:
        reasons.append("too_little_text")

    if chars_per_page is not None and chars_per_page < MIN_PDF_CHARS_PER_PAGE:
        reasons.append("low_text_per_page")

    if bad_marker_count >= MAX_BAD_ENCODING_MARKERS:
        reasons.append("encoding_corruption")

    if fragment_ratio > MAX_ONE_CHAR_TOKEN_RATIO:
        reasons.append("fragmented_text")

    return {
        "passes": not reasons,
        "reasons": reasons,
        "n_chars": n_chars,
        "n_pages": n_pages,
    }

## 3. PDF extractors

The extraction functions return text and the number of PDF pages. They do not decide which
method is “best”. Selection is handled separately by the fixed fallback sequence.

In [4]:
def extract_pdf_pymupdf(path: Path) -> tuple[str, int | None]:
    page_text: list[str] = []

    with fitz.open(path) as document:
        for page in document:
            page_text.append(page.get_text("text", sort=True) or "")
        n_pages = len(document)

    return clean_text("\n\n".join(page_text)), n_pages


def extract_pdf_pdftotext(path: Path) -> tuple[str, int | None]:
    if PDFTOTEXT_CLI is None:
        raise RuntimeError("pdftotext is not available on PATH.")

    with tempfile.TemporaryDirectory(prefix="pdftotext_") as tmpdir:
        output_path = Path(tmpdir) / "output.txt"

        command = [
            PDFTOTEXT_CLI,
            "-layout",
            "-enc",
            "UTF-8",
            str(path),
            str(output_path),
        ]

        completed = subprocess.run(
            command,
            capture_output=True,
            text=True,
            timeout=300,
            check=False,
        )

        if completed.returncode != 0:
            raise RuntimeError(completed.stderr[-2000:] or "pdftotext failed")

        text = output_path.read_text(encoding="utf-8", errors="replace")

    try:
        with fitz.open(path) as document:
            n_pages = len(document)
    except Exception:
        n_pages = None

    return clean_text(text), n_pages


def run_mineru(input_pdf: Path, output_dir: Path) -> None:
    if MINERU_CLI is None:
        raise RuntimeError("MinerU is not available on PATH.")

    try:
        with fitz.open(input_pdf) as document:
            n_pages = len(document)
    except Exception:
        n_pages = None

    if n_pages is None:
        timeout_seconds = OCR_TIMEOUT_SECONDS
    else:
        timeout_seconds = min(
            OCR_TIMEOUT_CAP_SECONDS,
            max(OCR_TIMEOUT_SECONDS, n_pages * OCR_SECONDS_PER_PAGE),
        )

    command = [
        MINERU_CLI,
        "-p",
        str(input_pdf),
        "-o",
        str(output_dir),
    ]

    if MINERU_PREFER_PIPELINE_BACKEND:
        command.extend(["-b", "pipeline"])

    if MINERU_FORCE_OCR_MODE:
        command.extend(["-m", "ocr"])

    environment = os.environ.copy()
    environment.setdefault("ORT_LOG_SEVERITY_LEVEL", "3")

    # MinerU has internal limits in addition to Python's subprocess timeout.
    # Raise those limits for long image-only PDFs.
    environment["MINERU_PDF_RENDER_TIMEOUT"] = str(
        MINERU_PDF_RENDER_TIMEOUT_SECONDS
    )
    environment["MINERU_LOCAL_API_STARTUP_TIMEOUT_SECONDS"] = str(
        MINERU_LOCAL_API_STARTUP_TIMEOUT_SECONDS
    )
    environment["MINERU_TASK_RESULT_TIMEOUT_SECONDS"] = str(
        MINERU_TASK_RESULT_TIMEOUT_SECONDS
    )
    environment["MINERU_PROCESSING_WINDOW_SIZE"] = str(
        MINERU_PROCESSING_WINDOW_SIZE
    )

    try:
        completed = subprocess.run(
            command,
            capture_output=True,
            text=True,
            timeout=timeout_seconds,
            env=environment,
            check=False,
        )
    except subprocess.TimeoutExpired as exc:
        page_label = n_pages if n_pages is not None else "unknown"
        raise RuntimeError(
            f"MinerU timed out after {timeout_seconds} seconds "
            f"for a {page_label}-page PDF."
        ) from exc

    if completed.returncode != 0:
        stderr_tail = (completed.stderr or "")[-4000:].strip()
        stdout_tail = (completed.stdout or "")[-2000:].strip()
        detail = stderr_tail or stdout_tail or "MinerU failed without output."
        raise RuntimeError(
            f"MinerU failed with return code {completed.returncode}. "
            f"Pages: {n_pages if n_pages is not None else 'unknown'}. "
            f"Timeout allowance: {timeout_seconds}s. "
            f"Output tail: {detail}"
        )

def find_mineru_text(output_dir: Path) -> Path | None:
    markdown = [
        path
        for path in output_dir.glob("**/*.md")
        if path.is_file() and path.stat().st_size > 0
    ]
    if markdown:
        return max(markdown, key=lambda path: path.stat().st_size)

    text_files = [
        path
        for path in output_dir.glob("**/*.txt")
        if path.is_file() and path.stat().st_size > 0
    ]
    if text_files:
        return max(text_files, key=lambda path: path.stat().st_size)

    return None


def extract_pdf_mineru(path: Path) -> tuple[str, int | None]:
    with tempfile.TemporaryDirectory(prefix="mineru_") as tmpdir:
        output_dir = Path(tmpdir)
        run_mineru(path, output_dir)

        text_path = find_mineru_text(output_dir)
        if text_path is None:
            raise RuntimeError(
                "MinerU completed but produced no non-empty Markdown/text file."
            )

        text = text_path.read_text(encoding="utf-8", errors="replace")

    try:
        with fitz.open(path) as document:
            n_pages = len(document)
    except Exception:
        n_pages = None

    return clean_text(text), n_pages


def try_extractor(path: Path, method: str, extractor) -> tuple[dict[str, Any] | None, str]:
    try:
        text, n_pages = extractor(path)
        assessment = assess_text(text, n_pages)

        return {
            "method": method,
            "text": text,
            "n_pages": n_pages,
            "assessment": assessment,
        }, ""

    except Exception as exc:
        return None, repr(exc)

## 4. Deterministic PDF fallback

The first extraction that passes the technical checks is accepted immediately.

OCR is therefore not run for every document. It is used only when the two native-text
methods both fail or remain technically suspicious.

There is no separate `review` state. If all available methods remain suspicious but a
non-empty extraction exists, the last available text is still retained as a successful
processing result. Only a complete failure to produce text is marked `failed`.

In [5]:
def process_pdf(path: Path) -> dict[str, Any]:
    attempts: list[dict[str, Any]] = []
    errors: list[str] = []
    fallback_reason: list[str] = []

    # 1. Default native extraction
    pymupdf_result, error = try_extractor(
        path,
        "pymupdf",
        extract_pdf_pymupdf,
    )

    if error:
        errors.append(f"pymupdf: {error}")
        fallback_reason.append("pymupdf_failed")
    elif pymupdf_result is not None:
        attempts.append(pymupdf_result)

        if pymupdf_result["assessment"]["passes"]:
            return {
                "selected": pymupdf_result,
                "status": "success",
                "ocr_used": False,
                "fallback_reason": "",
                "errors": errors,
            }

        fallback_reason.extend(
            f"pymupdf:{reason}"
            for reason in pymupdf_result["assessment"]["reasons"]
        )

    # 2. Cheap native fallback
    if RUN_PDFTOTEXT and PDFTOTEXT_CLI is not None:
        poppler_result, error = try_extractor(
            path,
            "pdftotext",
            extract_pdf_pdftotext,
        )

        if error:
            errors.append(f"pdftotext: {error}")
            fallback_reason.append("pdftotext_failed")
        elif poppler_result is not None:
            attempts.append(poppler_result)

            if poppler_result["assessment"]["passes"]:
                return {
                    "selected": poppler_result,
                    "status": "success",
                    "ocr_used": False,
                    "fallback_reason": "; ".join(fallback_reason),
                    "errors": errors,
                }

            fallback_reason.extend(
                f"pdftotext:{reason}"
                for reason in poppler_result["assessment"]["reasons"]
            )

    elif RUN_PDFTOTEXT:
        fallback_reason.append("pdftotext_unavailable")

    # 3. OCR fallback
    if RUN_OCR and MINERU_CLI is not None:
        ocr_result, error = try_extractor(
            path,
            "mineru_ocr",
            extract_pdf_mineru,
        )

        if error:
            errors.append(f"mineru_ocr: {error}")
            fallback_reason.append("mineru_failed")
        elif ocr_result is not None and ocr_result["text"].strip():
            attempts.append(ocr_result)

            # OCR is the final fallback. Keep non-empty OCR output even if one
            # of the simple technical thresholds still triggers.
            return {
                "selected": ocr_result,
                "status": "success",
                "ocr_used": True,
                "fallback_reason": "; ".join(fallback_reason),
                "errors": errors,
            }

    elif RUN_OCR:
        fallback_reason.append("mineru_unavailable")

    # If OCR is unavailable or fails, retain the latest non-empty native result.
    nonempty_attempts = [
        attempt
        for attempt in attempts
        if attempt.get("text", "").strip()
    ]

    if nonempty_attempts:
        selected = nonempty_attempts[-1]
        return {
            "selected": selected,
            "status": "success",
            "ocr_used": selected["method"] == "mineru_ocr",
            "fallback_reason": "; ".join(fallback_reason),
            "errors": errors,
        }

    return {
        "selected": None,
        "status": "failed",
        "ocr_used": False,
        "fallback_reason": "; ".join(fallback_reason),
        "errors": errors,
    }

## 5. Resumable JSONL processing state

`logs/com_dbcomp/db_comp_processing_state.jsonl` is an operational processing tracker,
not a publication data table.

One state record is appended after each document is processed. On a later run, a successful
record can be reused when:

- the raw PDF SHA-256 is unchanged;
- the processing version is unchanged;
- the processed text file still exists; and
- its SHA-256 matches the stored text checksum.

Failed documents are retried automatically. The actual files remain authoritative: a JSONL
record alone never causes a missing or changed text file to be treated as complete.

Set `FORCE_REPROCESS = True` to deliberately rebuild all text files.

In [6]:
def json_safe(value: Any) -> Any:
    if value is None:
        return None
    if isinstance(value, (str, int, float, bool)):
        if isinstance(value, float) and pd.isna(value):
            return None
        return value
    if isinstance(value, Path):
        return value.as_posix()
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [json_safe(item) for item in value]
    if pd.isna(value):
        return None
    return str(value)


def load_processing_state() -> dict[str, dict[str, Any]]:
    """
    Read the JSONL tracker and keep the latest record for each DB-COMP file URL.
    A malformed partial line is ignored so an interrupted write does not block restart.
    """
    latest: dict[str, dict[str, Any]] = {}

    if not PROCESSING_STATE_JSONL.exists():
        return latest

    with PROCESSING_STATE_JSONL.open("r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if not line:
                continue

            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                continue

            file_url = str(record.get("file_url", "")).strip()
            if file_url:
                latest[file_url] = record

    return latest


def append_processing_state(record: dict[str, Any]) -> None:
    with PROCESSING_STATE_JSONL.open("a", encoding="utf-8") as handle:
        handle.write(
            json.dumps(
                json_safe(record),
                ensure_ascii=False,
                sort_keys=True,
            )
            + "\n"
        )


def expected_text_path(text_file_name: str) -> Path:
    return PROCESSED_TEXT_DIR / Path(text_file_name).name


def state_is_reusable(
    state_record: dict[str, Any] | None,
    *,
    raw_sha256: str,
    text_path: Path,
) -> bool:
    if FORCE_REPROCESS or not state_record:
        return False

    if state_record.get("processing_status") != "success":
        return False

    if state_record.get("processing_version") != PROCESSING_VERSION:
        return False

    if str(state_record.get("raw_sha256", "")).lower() != raw_sha256.lower():
        return False

    expected_portable_text = repo_path(text_path)
    if state_record.get("text_path") != expected_portable_text:
        return False

    if not text_path.exists() or text_path.stat().st_size <= 0:
        return False

    text_sha256 = str(state_record.get("text_sha256", "")).lower()
    if not re.fullmatch(r"[0-9a-f]{64}", text_sha256):
        return False

    return sha256_file(text_path) == text_sha256


def process_document(input_row: dict[str, Any], raw_sha256: str) -> dict[str, Any]:
    raw_path = Path(input_row["raw_path_internal"])
    text_path = expected_text_path(input_row["text_file_name"])

    result = process_pdf(raw_path)
    selected = result["selected"]

    base = {
        "timestamp_utc": utc_now_iso(),
        "processing_version": PROCESSING_VERSION,
        "file_url": input_row["file_url"],
        "file_name": input_row["file_name"],
        "raw_path": repo_path(raw_path),
        "raw_sha256": raw_sha256,
    }

    if selected is None:
        return {
            **base,
            "processing_status": "failed",
            "extraction_method": "",
            "ocr_used": False,
            "fallback_reason": result.get("fallback_reason", ""),
            "text_path": "",
            "text_sha256": "",
            "n_pages": None,
            "n_chars": 0,
            "error": " | ".join(result.get("errors", [])) or "no_text_extracted",
        }

    text = selected["text"]
    text_path.write_text(text, encoding="utf-8")
    text_sha256 = sha256_file(text_path)

    return {
        **base,
        "processing_status": "success",
        "extraction_method": selected["method"],
        "ocr_used": bool(result["ocr_used"]),
        "fallback_reason": result.get("fallback_reason", ""),
        "text_path": repo_path(text_path),
        "text_sha256": text_sha256,
        "n_pages": selected["assessment"]["n_pages"],
        "n_chars": selected["assessment"]["n_chars"],
        "error": "",
    }

## 6. Process the DB-COMP collection

`tqdm` provides the live progress display. The JSONL file provides persistent restart state.

The progress bar reports successful documents, failures, OCR use and state records reused
from an earlier run.

In [7]:
processing_state = load_processing_state()
manifest_rows: list[dict[str, Any]] = []

run_counts = {
    "success": 0,
    "failed": 0,
    "ocr": 0,
    "reused": 0,
}

target_rows = work_manifest.to_dict("records")

progress = tqdm(
    target_rows,
    desc="Processing DB-COMP PDFs",
    unit="doc",
    dynamic_ncols=True,
)

for input_row in progress:
    raw_path = Path(input_row["raw_path_internal"])
    raw_sha256 = sha256_file(raw_path)
    text_path = expected_text_path(input_row["text_file_name"])

    previous_state = processing_state.get(input_row["file_url"])

    if state_is_reusable(
        previous_state,
        raw_sha256=raw_sha256,
        text_path=text_path,
    ):
        state_record = previous_state
        run_counts["reused"] += 1

    else:
        state_record = process_document(input_row, raw_sha256)
        append_processing_state(state_record)
        processing_state[input_row["file_url"]] = state_record

    status = state_record.get("processing_status", "failed")
    if status == "success":
        run_counts["success"] += 1
    else:
        run_counts["failed"] += 1

    if state_record.get("ocr_used"):
        run_counts["ocr"] += 1

    manifest_rows.append(
        {
            "source": input_row["source"],
            "dbcomp_document_id": input_row["dbcomp_document_id"],
            "case_number": input_row["case_number"],
            "case_number_source": input_row["case_number_source"],
            "case_type": input_row["case_type"],
            "dbcomp_date": input_row["dbcomp_date"],
            "document_name": input_row["document_name"],
            "file_url": input_row["file_url"],
            "file_name": input_row["file_name"],
            "raw_path": state_record.get("raw_path", repo_path(raw_path)),
            "raw_sha256": state_record.get("raw_sha256", raw_sha256),
            "text_path": state_record.get("text_path", ""),
            "text_sha256": state_record.get("text_sha256", ""),
            "extraction_method": state_record.get("extraction_method", ""),
            "ocr_used": state_record.get("ocr_used", False),
            "processing_status": status,
            "fallback_reason": state_record.get("fallback_reason", ""),
            "n_pages": state_record.get("n_pages"),
            "n_chars": state_record.get("n_chars", 0),
            "error": state_record.get("error", ""),
        }
    )

    progress.set_postfix(
        {
            "ok": run_counts["success"],
            "fail": run_counts["failed"],
            "ocr": run_counts["ocr"],
            "reused": run_counts["reused"],
        },
        refresh=False,
    )


processing_manifest = pd.DataFrame(manifest_rows)

print(f"Documents considered: {len(processing_manifest):,}")
print(f"Successful: {run_counts['success']:,}")
print(f"Failed: {run_counts['failed']:,}")
print(f"OCR selected: {run_counts['ocr']:,}")
print(f"Checkpoints reused this run: {run_counts['reused']:,}")

Processing DB-COMP PDFs:   0%|          | 0/837 [00:00<?, ?doc/s]

Documents considered: 837
Successful: 837
Failed: 0
OCR selected: 19
Checkpoints reused this run: 835


## 7. Save the compact processing manifest

The processing manifest links each DB-COMP source PDF to its canonical text file and records
the extraction method, checksums and the minimum information needed to understand fallback
behaviour.

Detailed per-method candidate scores, alternative text copies, Excel workbooks and separate
failure manifests are intentionally not produced.

In [8]:
processing_manifest.to_csv(
    TEXT_MANIFEST_CSV,
    index=False,
    encoding="utf-8",
)

print("Saved: output/com_dbcomp/db_comp_text_manifest.csv")

display(
    processing_manifest[
        [
            "dbcomp_document_id",
            "case_number",
            "file_name",
            "text_path",
            "extraction_method",
            "ocr_used",
            "processing_status",
        ]
    ].head()
)

failed = processing_manifest.loc[
    processing_manifest["processing_status"] == "failed"
].copy()

if len(failed):
    print()
    print(f"Failed documents: {len(failed):,}")
    display(
        failed[
            [
                "dbcomp_document_id",
                "case_number",
                "file_name",
                "fallback_reason",
                "error",
            ]
        ].head(20)
    )

Saved: output/com_dbcomp/db_comp_text_manifest.csv


,dbcomp_document_id,case_number,file_name,text_path,extraction_method,ocr_used,processing_status
0,,39904,AT_39904_5942450_1110_7.pdf,ec_cjeu/data/processed/com_dbcomp/text/AT_3990...,pymupdf,False,success
1,,40135,AT_40135_8399795_5456_9.pdf,ec_cjeu/data/processed/com_dbcomp/text/AT_4013...,pymupdf,False,success
2,,40136,AT_40136_7970992_6095_3.pdf,ec_cjeu/data/processed/com_dbcomp/text/AT_4013...,pymupdf,False,success
3,,40490,AT_40490_8263501_702_3.pdf,ec_cjeu/data/processed/com_dbcomp/text/AT_4049...,pymupdf,False,success
4,,40522,AT_40522_8986595_2868_7.pdf,ec_cjeu/data/processed/com_dbcomp/text/AT_4052...,pymupdf,False,success


## 8. Technical validation and reproducibility

The automated checks in this notebook are intentionally limited to identifying obvious
technical extraction problems and deciding whether a fallback method should be attempted.
They are not presented as a measure of document quality or legal content.

For the final Scientific Data release, the processing description should report the fixed
fallback hierarchy and thresholds used here. A practical human validation step is to inspect
the OCR-processed documents and a small sample of ordinary native-text extractions across the
time span of the collection, confirming that the processed text corresponds to the source
PDF and remains broadly readable.

The raw PDFs are never modified. `raw_sha256` and `text_sha256` identify the exact input and
processed files used in this stage. The processing version and JSONL state make interrupted
runs reproducible without adding development-time audit tables to the publication outputs.

External software requirements are PyMuPDF, Poppler (`pdftotext`) and MinerU. Python package
versions should be frozen at repository release level (for example in `requirements.txt` or
an environment file) rather than repeated in every manifest row.

## Output

This notebook produces:

- `ec_cjeu/data/processed/com_dbcomp/text/<source-filename>.txt` for filenames that are unique;
- `ec_cjeu/data/processed/com_dbcomp/text/<source-filename>__<url-hash>.txt` when source filenames collide;
- `ec_cjeu/output/com_dbcomp/db_comp_text_manifest.csv`;
- `ec_cjeu/logs/com_dbcomp/db_comp_processing_state.jsonl`.

The processing manifest contains portable raw and processed paths such as:

`ec_cjeu/data/raw/com_dbcomp/files/example.pdf`

and either

`ec_cjeu/data/processed/com_dbcomp/text/example.txt`

or, for a colliding source filename,

`ec_cjeu/data/processed/com_dbcomp/text/example__a1b2c3d4e5f6.txt`.
